In [1]:
import os
os.chdir("/kaggle/input/tweets")
print("starting")

starting


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW 
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report # Added for testing
import numpy as np
import os

# --- 1. Data Parsing ---
LABEL_FILES = ['label.txt', 'label_1.txt']
SOURCE_FILES = ['source_tweets.txt', 'source_tweets_1.txt']

def load_and_merge_data(label_files, source_files):
    id_to_label = {}
    # Parse labels
    for f_path in label_files:
        if os.path.exists(f_path):
            with open(f_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split(':')
                    if len(parts) >= 2:
                        id_to_label[parts[-1]] = parts[0]

    data = []
    # Parse tweets
    for f_path in source_files:
        if os.path.exists(f_path):
            with open(f_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        t_id = parts[0]
                        text = " ".join(parts[1:])
                        if t_id in id_to_label:
                            label = id_to_label[t_id]
                            if label in ['true', 'false', 'unverified', 'non-rumor']:
                                data.append({'text': text, 'label': label})
    
    print(f"Data Loaded: {len(data)} matched samples.")
    return data

# --- 2. Dataset & Model ---
class FakeNewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_map = label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = str(item['text'])
        label = self.label_map[item['label']]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class BERTClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        return self.classifier(self.dropout(pooled_output))

# --- 3. Evaluation Function (TESTING) ---
def evaluate_model(model, data_loader, device, label_map):
    model.eval() # Set to evaluation mode (stops updating weights)
    
    predictions = []
    actual_labels = []

    with torch.no_grad(): # Disable gradient calculation for testing
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, mask)
            _, preds = torch.max(outputs, dim=1)

            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    # Calculate metrics
    acc = accuracy_score(actual_labels, predictions)
    
    # Map IDs back to names for the report
    target_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    report = classification_report(actual_labels, predictions, target_names=target_names, zero_division=0)
    
    return acc, report

# --- 4. Main Execution ---
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load Data
    raw_data = load_and_merge_data(LABEL_FILES, SOURCE_FILES)
    if not raw_data:
        print("No data found. Creating dummy data...")
        raw_data = [{'text': 'Dummy tweet', 'label': 'true'}] * 10
    
    label_map = {'true': 0, 'false': 1, 'unverified': 2, 'non-rumor': 3}
    
    # --- SPLITTING DATA ---
    # 90% Training Data, 10% Testing Data
    df_train, df_test = train_test_split(raw_data, test_size=0.1, random_state=42)
    print(f"Training on {len(df_train)} samples. Testing on {len(df_test)} samples.")
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    # Create DataLoaders
    # train_loader is for TRAINING (shuffled)
    train_loader = DataLoader(FakeNewsDataset(df_train, tokenizer, label_map=label_map), batch_size=16, shuffle=True)
    # val_loader is for TESTING (not shuffled)
    val_loader = DataLoader(FakeNewsDataset(df_test, tokenizer, label_map=label_map), batch_size=16)

    model = BERTClassifier(len(label_map)).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss().to(device)
    
    EPOCHS = 5 # Reduced to 5. 20 is usually too many for BERT and causes overfitting.
    
    print("\nStarting training and testing loop...")
    print("=" * 60)
    
    for epoch in range(EPOCHS):
        # --- TRAIN PHASE ---
        model.train() # Enable dropout and backprop
        total_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, mask)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train_loss = total_loss / len(train_loader)
        
        # --- TEST PHASE ---
        # We test on the 'val_loader' (test set) immediately after training
        val_acc, val_report = evaluate_model(model, val_loader, device, label_map)
        
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Training Loss: {avg_train_loss:.4f}")
        print(f"Test Accuracy: {val_acc:.4f}")
        print("Test Report:")
        print(val_report)
        print("-" * 60)

if __name__ == "__main__":
    main()

2026-01-02 17:22:34.167408: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767374554.648789      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767374554.764078      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767374555.881646      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767374555.881692      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767374555.881695      24 computation_placer.cc:177] computation placer alr

Using device: cuda
Data Loaded: 2308 matched samples.
Training on 2077 samples. Testing on 231 samples.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


Starting training and testing loop...
Epoch 1/5
Training Loss: 1.2600
Test Accuracy: 0.5411
Test Report:
              precision    recall  f1-score   support

        true       0.68      0.86      0.76        51
       false       0.53      0.30      0.38        64
  unverified       0.85      0.34      0.49        67
   non-rumor       0.38      0.80      0.51        49

    accuracy                           0.54       231
   macro avg       0.61      0.57      0.54       231
weighted avg       0.62      0.54      0.52       231

------------------------------------------------------------
Epoch 2/5
Training Loss: 0.7574
Test Accuracy: 0.7359
Test Report:
              precision    recall  f1-score   support

        true       0.67      0.94      0.78        51
       false       0.86      0.67      0.75        64
  unverified       0.87      0.69      0.77        67
   non-rumor       0.59      0.67      0.63        49

    accuracy                           0.74       231
   ma

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import re
import os

# --- 1. Clean Text Function ---
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\burl\b', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- 2. Data Loading ---
LABEL_FILES = ['label.txt', 'label_1.txt']
SOURCE_FILES = ['source_tweets.txt', 'source_tweets_1.txt']

def load_and_merge_data(label_files, source_files):
    id_to_label = {}
    for f_path in label_files:
        if os.path.exists(f_path):
            with open(f_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split(':')
                    if len(parts) >= 2:
                        id_to_label[parts[-1]] = parts[0]
    data = []
    for f_path in source_files:
        if os.path.exists(f_path):
            with open(f_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        t_id = parts[0]
                        raw_text = " ".join(parts[1:])
                        if t_id in id_to_label:
                            label = id_to_label[t_id]
                            if label in ['true', 'false', 'unverified', 'non-rumor']:
                                cleaned = clean_text(raw_text)
                                if len(cleaned) > 5:
                                    data.append({'text': cleaned, 'label': label})
    print(f"Data Loaded: {len(data)} samples")
    return data

# --- 3. Model Classes ---
class FakeNewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_map = label_map
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        label = self.label_map[item['label']]
        enc = self.tokenizer.encode_plus(text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'labels': torch.tensor(label, dtype=torch.long)}

class RoBERTaClassifier(nn.Module):
    def __init__(self, num_classes):
        super(RoBERTaClassifier, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(self.roberta.config.hidden_size, num_classes)
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.out(self.drop(outputs.last_hidden_state[:, 0, :]))

# --- 4. Main Training Routine ---
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Load Data
    raw_data = load_and_merge_data(LABEL_FILES, SOURCE_FILES)
    if not raw_data: 
        print("ERROR: No data found.")
        return

    label_map = {'true': 0, 'false': 1, 'unverified': 2, 'non-rumor': 3}
    df_train, df_test = train_test_split(raw_data, test_size=0.1, random_state=42, stratify=[d['label'] for d in raw_data])
    
    # Class weights for imbalance
    y_train = [label_map[d['label']] for d in df_train]
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(cw, dtype=torch.float).to(device)

    # Setup
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    train_loader = DataLoader(FakeNewsDataset(df_train, tokenizer, label_map=label_map), batch_size=16, shuffle=True)
    
    model = RoBERTaClassifier(len(label_map)).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    print("Starting Training (RoBERTa)...")
    model.train()
    
    # Training Loop
    for epoch in range(4):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} done. Loss: {total_loss/len(train_loader):.4f}")

    # --- FAIL-SAFE SAVING ---
    print("\nAttempting to save model...")
    save_filename = 'roberta_model.bin'
    
    try:
        # Try saving to current working directory
        current_path = os.path.join(os.getcwd(), save_filename)
        torch.save(model.state_dict(), current_path)
        print(f"SUCCESS: Model saved to: {current_path}")
        final_path = current_path
    except Exception as e:
        print(f"Could not save to current directory: {e}")
        print("Attempting to save to /tmp/...")
        try:
            # Fallback to /tmp which is usually writable
            tmp_path = os.path.join('/tmp', save_filename)
            torch.save(model.state_dict(), tmp_path)
            print(f"SUCCESS: Model saved to: {tmp_path}")
            final_path = tmp_path
        except Exception as e2:
            print(f"CRITICAL ERROR: Could not save model anywhere. {e2}")
            return

    # Return path so we can load it later
    return final_path

if __name__ == "__main__":
    saved_model_path = main()

Using device: cuda
Data Loaded: 2306 samples


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting Training (RoBERTa)...
Epoch 1 done. Loss: 1.2307
Epoch 2 done. Loss: 0.6836
Epoch 3 done. Loss: 0.3543
Epoch 4 done. Loss: 0.1589

Attempting to save model...
Could not save to current directory: File /kaggle/input/tweets/roberta_model.bin cannot be opened.
Attempting to save to /tmp/...
SUCCESS: Model saved to: /tmp/roberta_model.bin


In [4]:
# --- EVALUATION CODE ---
import torch
from sklearn.metrics import classification_report, accuracy_score

# REPLACE THIS WITH THE PATH PRINTED ABOVE IF DIFFERENT
MODEL_PATH = '/tmp/roberta_model.bin' 
# OR just 'roberta_model.bin' if it saved locally

if 'saved_model_path' in globals() and saved_model_path:
    MODEL_PATH = saved_model_path

def evaluate_saved_model(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    label_map = {'true': 0, 'false': 1, 'unverified': 2, 'non-rumor': 3}
    
    # Re-initialize architecture
    model = RoBERTaClassifier(len(label_map))
    
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model from {model_path}")
    except FileNotFoundError:
        print(f"File not found at {model_path}. Please check the path.")
        return

    model.to(device)
    model.eval()

    # Load Test Data
    raw_data = load_and_merge_data(LABEL_FILES, SOURCE_FILES)
    _, df_test = train_test_split(raw_data, test_size=0.1, random_state=42, stratify=[d['label'] for d in raw_data])
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    test_loader = DataLoader(FakeNewsDataset(df_test, tokenizer, label_map=label_map), batch_size=16)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, mask)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    target_names = [k for k, v in sorted(label_map.items(), key=lambda x: x[1])]
    print("\n--- Final Accuracy Report ---")
    print(classification_report(all_labels, all_preds, target_names=target_names))
    print(f"Accuracy Score: {accuracy_score(all_labels, all_preds):.4f}")

if __name__ == "__main__":
    evaluate_saved_model(MODEL_PATH) 

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model from /tmp/roberta_model.bin
Data Loaded: 2306 samples

--- Final Accuracy Report ---
              precision    recall  f1-score   support

        true       0.90      0.90      0.90        58
       false       0.95      0.61      0.74        57
  unverified       0.84      0.91      0.88        58
   non-rumor       0.74      0.93      0.82        58

    accuracy                           0.84       231
   macro avg       0.86      0.84      0.84       231
weighted avg       0.86      0.84      0.84       231

Accuracy Score: 0.8398


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Text Preprocessor 
class TextPreprocessor:
    def __init__(self, model_name='roberta-base', max_len=512):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_len = max_len

    def __call__(self, text):
        text = str(text).strip().replace('\n', ' ').replace('\r', ' ')
        
        encoding = self.tokenizer(
            text, add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

# Vision Preprocessor
class VisionPreprocessor:
    def __init__(self, img_size=224, fps=1.5, face_pad=0.1):
        self.img_size = img_size
        self.fps = fps
        self.face_pad = face_pad
        self.face_detector = mp.solutions.face_detection.FaceDetection(
            model_selection=1, min_detection_confidence=0.5)

    def _extract_faces(self, frame):
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.face_detector.process(rgb_frame)
        crops = []
        if results.detections:
            for det in results.detections:
                bbox = det.location_data.relative_bounding_box
                x, y, bw, bh = int(bbox.xmin*w), int(bbox.ymin*h), int(bbox.width*w), int(bbox.height*h)
                pad_x, pad_y = int(bw*self.face_pad), int(bh*self.face_pad)
                crop = frame[max(0,y-pad_y):min(h,y+bh+pad_y), max(0,x-pad_x):min(w,x+bw+pad_x)]
                if crop.size > 0:
                    crops.append(cv2.resize(crop, (self.img_size, self.img_size)))
        return crops

    def __call__(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return np.zeros((0, 3, self.img_size, self.img_size), dtype=np.float32)
        fps_vid = cap.get(cv2.CAP_PROP_FPS)
        if fps_vid <= 0:
            fps_vid = 30.0
        frame_interval = max(1, int(fps_vid / self.fps))
        frames, count = [], 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if count % frame_interval == 0:
                crops = self._extract_faces(frame)
                frames.extend(crops)
            count += 1
        cap.release()
        if len(frames) == 0:
            return np.zeros((0, 3, self.img_size, self.img_size), dtype=np.float32)
        frames = np.stack(frames).astype(np.float32) / 255.0
        return np.transpose(frames, (0, 3, 1, 2))

In [ ]:
class TextBranch(nn.Module):
    def __init__(self, model_name='roberta-base', dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1), nn.Sigmoid())
        for param in self.backbone.embeddings.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(outputs.last_hidden_state[:, 0, :]).squeeze(-1)

class VisionBranch(nn.Module):
    def __init__(self, model_name='google/vit-base-patch16-224', dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.norm = nn.LayerNorm(768)
        self.classifier = nn.Sequential(nn.Linear(768, 1), nn.Sigmoid())

    def forward(self, frames):
        if frames.shape == 0:
            return torch.tensor([], device=frames.device)
        outputs = self.backbone(pixel_values=frames)
        normed = self.norm(outputs.last_hidden_state[:, 0, :])
        return self.classifier(normed).squeeze(-1)

class MultimodalFakeNewsDetector(nn.Module):
    def __init__(self, alpha=0.5, threshold=0.5):
        super().__init__()
        self.text_branch = TextBranch()
        self.vision_branch = VisionBranch()
        self.alpha = alpha
        self.threshold = threshold

    def forward(self, input_ids, attention_mask, all_frames, frame_counts):
        p_text = self.text_branch(input_ids, attention_mask)
        p_vision = torch.zeros_like(p_text)
        if all_frames.shape[0] > 0:
            p_vision_frames = self.vision_branch(all_frames)
            start = 0
            for i, count in enumerate(frame_counts):
                if count > 0:
                    p_vision[i] = p_vision_frames[start:start+count].mean()
                else:
                    p_vision[i] = 0.5
                start += count
        else:
            p_vision.fill_(0.5)
        p_final = self.alpha * p_text + (1 - self.alpha) * p_vision
        return p_final, p_text, p_vision

# Training Loop
def train_epoch(model, loader, opt_text, opt_vision, device):
    model.train()
    total_loss = 0
    criterion = nn.BCELoss()
    for batch in tqdm(loader, desc='Training', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        frames = batch['all_frames'].to(device)
        frame_counts = batch['frame_counts'].to(device)
        labels = batch['labels'].to(device)
        
        p_final, _, _ = model(input_ids, attention_mask, frames, frame_counts)
        loss = criterion(p_final, labels)
        opt_text.zero_grad()
        opt_vision.zero_grad()
        loss.backward()
        opt_text.step()
        opt_vision.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Fusion Coefficient Calibration
def calibrate_fusion_weights(model, val_loader, device):
    model.eval()
    alpha_candidates = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    threshold_candidates = [0.40, 0.45, 0.50, 0.55, 0.60]
    best_f1, best_alpha, best_tau = 0.0, 0.4, 0.5
    all_p_text, all_p_vision, all_labels = [], [], []
    for batch in tqdm(val_loader, desc='Calibration'):
        ...  # extract p_text, p_vision per batch
    all_p_text = np.array(all_p_text)
    all_p_vision = np.array(all_p_vision)
    all_labels = np.array(all_labels)
    for alpha in alpha_candidates:
        for tau in threshold_candidates:
            p_final = alpha*all_p_text + (1-alpha)*all_p_vision
            binary_preds = (p_final > tau).astype(int)
            current_f1 = f1_score(all_labels, binary_preds, zero_division=0)
            if current_f1 > best_f1:
                best_f1, best_alpha, best_tau = current_f1, alpha, tau
    model.alpha = best_alpha
    model.threshold = best_tau
    print(f'Optimal alpha={best_alpha}, tau={best_tau} (Val F1: {best_f1:.4f})')
    return model